# CPU GNN Experiment: Graph-Conditioned Configuration Ranker

This notebook targets `layout:xla:default` and `layout:xla:random`. Keep the other
three collections from the main baseline when evaluating a combined submission.

The first run extracts compressed configuration arrays to `.gnn_cache` once.
Later batches use memory-mapped arrays and cached static graph tensors. Extraction
requires disk space approximately equal to the uncompressed configuration arrays;
delete the cache when it is no longer needed. It is local to the current runtime.

Start with the default three epochs to measure speed. Inspect `gnn_runs/*/history.csv`
before raising the epoch limit. Best validation checkpoints are saved as `best.pt`.
These results are diagnostics, not measured Kaggle leaderboard improvements.


## 1. Download Kaggle Data in Colab

This block is copied from the executed Colab workflow. It expects `KAGGLE_USERNAME` and `KAGGLE_KEY` to exist in Colab Secrets.


In [ ]:
from google.colab import userdata

username = userdata.get("KAGGLE_USERNAME")
key = userdata.get("KAGGLE_KEY")

print("Username exists:", username is not None)
print("Key exists:", key is not None)
from google.colab import userdata
import os

import kagglehub

# Get Kaggle credentials from Colab Secrets.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Check that credentials were loaded.
print("Username loaded:", bool(os.environ.get("KAGGLE_USERNAME")))
print("Key loaded:", bool(os.environ.get("KAGGLE_KEY")))

# Download directly into /content so it is visible in Colab Files.
path = kagglehub.competition_download(
    "predict-ai-model-runtime",
    output_dir="/content/predict-ai-model-runtime"
)

print("Downloaded to:", path)

# Keep this output short. Detailed file counts are shown in the inspection section.
print("\nTop-level files/folders:")
for item in sorted(os.listdir(path)):
    print("-", item)


## 2. Imports and Configuration

The GNN uses plain PyTorch rather than PyTorch Geometric. This keeps installation simple and avoids heavy graph-library dependencies.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import os
import time
import gc
import warnings
import hashlib
import shutil
import zipfile
from collections import OrderedDict


def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("scikit-learn", "sklearn")
ensure_package("tqdm")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    raise RuntimeError("PyTorch is required for this GNN experiment notebook") from exc

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)

# Keep these conservative for Colab RAM. Increase only after a successful smoke test.
GNN_COLLECTIONS = ["layout:xla:default", "layout:xla:random"]
GNN_SUBGRAPH_HOPS = 2
GNN_MAX_SUBGRAPH_NODES = 512
GNN_MAX_TRAIN_FILES = 8
GNN_MAX_VALID_FILES = 5
GNN_MAX_TRAIN_CONFIGS_PER_FILE = 64
GNN_MAX_VALID_CONFIGS_PER_FILE = 1000
GNN_BATCH_SIZE = 16
GNN_PREDICT_BATCH_SIZE = 32
GNN_HIDDEN_DIM = 32
GNN_EPOCHS = 3
GNN_LR = 1e-3
PAIR_MARGIN = 0.1
PAIR_MIN_LOG_RUNTIME_GAP = 0.02
PAIR_MAX_PAIRS = 512

print("GNN_COLLECTIONS:", GNN_COLLECTIONS)
print("GNN_SUBGRAPH_HOPS:", GNN_SUBGRAPH_HOPS)
print("GNN_MAX_SUBGRAPH_NODES:", GNN_MAX_SUBGRAPH_NODES)
print("GNN_BATCH_SIZE:", GNN_BATCH_SIZE)

# Cap CPU threads to avoid oversubscription on small graph batches.
# Override this after benchmarking on your own CPU.
GNN_CPU_THREADS = min(4, os.cpu_count() or 1)
if DEVICE.type == "cpu":
    torch.set_num_threads(GNN_CPU_THREADS)

# Compressed configuration arrays are extracted once, then memory-mapped.
# This trades disk space for less repeated decompression and bounded RAM.
GNN_CACHE_DIR = Path.cwd() / ".gnn_cache"
GNN_GRAPH_CACHE_MB = 256
GNN_OUTPUT_DIR = Path.cwd() / "gnn_runs"
GNN_PATIENCE = 3
GNN_MIN_DELTA = 1e-4
GNN_VALIDATE_EVERY = 1
print("CPU threads:", torch.get_num_threads())
print("Training/inference batch sizes:", GNN_BATCH_SIZE, GNN_PREDICT_BATCH_SIZE)
print("Cache:", GNN_CACHE_DIR)


## 3. Locate Data

Only the two layout XLA collections are used here. The other collections stay in the main notebook.


In [ ]:
def find_data_root():
    candidates = [
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path("/content/data"),
        Path.cwd() / "predict-ai-model-runtime",
        Path.cwd().parent / "predict-ai-model-runtime",
        Path("/content/predict-ai-model-runtime"),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "npz_all" / "npz").exists():
            return candidate
    raise FileNotFoundError("Could not find npz_all/npz. Put the Kaggle data folder in data/ or update find_data_root().")


DATA_ROOT = find_data_root()
NPZ_ROOT = DATA_ROOT / "npz_all" / "npz"
COLLECTIONS = {
    "layout:xla:default": NPZ_ROOT / "layout" / "xla" / "default",
    "layout:xla:random": NPZ_ROOT / "layout" / "xla" / "random",
}

print("DATA_ROOT:", DATA_ROOT)
for name, path in COLLECTIONS.items():
    print(name, "train", len(list((path / "train").glob("*.npz"))), "valid", len(list((path / "valid").glob("*.npz"))))


def split_files(collection_name, split):
    return sorted((COLLECTIONS[collection_name] / split).glob("*.npz"))


## 4. Sampling and Validation Utilities

In [ ]:
def get_num_configs(data):
    if "node_config_feat" in data:
        return data["node_config_feat"].shape[0]
    if "config_feat" in data:
        return data["config_feat"].shape[0]
    raise KeyError("Could not find config features")


def choose_indices(n_items, max_items=None, seed=RANDOM_SEED):
    if max_items is None or n_items <= max_items:
        return np.arange(n_items, dtype=np.int64)
    local_rng = np.random.default_rng(seed)
    return np.sort(local_rng.choice(n_items, size=max_items, replace=False)).astype(np.int64)


def choose_runtime_stratified_indices(runtimes, max_items, seed=RANDOM_SEED):
    runtimes = np.asarray(runtimes, dtype=np.float64)
    valid_idx = np.flatnonzero(np.isfinite(runtimes) & (runtimes > 0))
    if max_items is None or len(valid_idx) <= max_items:
        return valid_idx
    if len(valid_idx) == 0:
        raise ValueError("No positive finite training runtimes")

    local_rng = np.random.default_rng(seed)
    sorted_idx = valid_idx[np.argsort(runtimes[valid_idx])]
    fastest_count = max(1, int(max_items * 0.20))
    slowest_count = max(1, int(max_items * 0.10))
    selected = set(sorted_idx[:fastest_count].tolist())
    selected.update(sorted_idx[-slowest_count:].tolist())

    remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
    budget = max_items - len(selected)
    if budget > 0 and len(remaining) > 0:
        chosen = local_rng.choice(remaining, size=min(budget, len(remaining)), replace=False)
        selected.update(chosen.tolist())
    return np.array(sorted(selected), dtype=np.int64)


def choose_config_indices(data, split, max_items=None, seed=RANDOM_SEED):
    n_items = get_num_configs(data)
    if split == "train" and "config_runtime" in data:
        return choose_runtime_stratified_indices(data["config_runtime"], max_items, seed=seed)
    return choose_indices(n_items, max_items=max_items, seed=seed)


def sampled_kendall_score(y_true, y_pred, max_pairs=20000, seed=RANDOM_SEED):
    n = len(y_true)
    if n < 2:
        return np.nan
    local_rng = np.random.default_rng(seed)
    i = local_rng.integers(0, n, size=max_pairs)
    j = local_rng.integers(0, n, size=max_pairs)
    mask = i != j
    i, j = i[mask], j[mask]
    true_order = np.sign(y_true[i] - y_true[j])
    pred_order = np.sign(y_pred[i] - y_pred[j])
    useful = true_order != 0
    if useful.sum() == 0:
        return np.nan
    # Prediction ties contribute zero rather than disappearing from the score.
    return float(np.mean(true_order[useful] * pred_order[useful]))


def centered_log_runtime(runtimes):
    log_runtime = np.log1p(np.asarray(runtimes, dtype=np.float64))
    return log_runtime - np.median(log_runtime)


## 5. K-Hop Subgraph Around Configurable Nodes

The subgraph starts from all `node_config_ids`, expands by `GNN_SUBGRAPH_HOPS`, and keeps one combined induced subgraph per graph file.


In [ ]:
def build_adjacency(edge_index, node_count):
    neighbors = [set() for _ in range(node_count)]
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < node_count and 0 <= dst < node_count:
            neighbors[int(src)].add(int(dst))
            neighbors[int(dst)].add(int(src))
    return neighbors


def khop_subgraph_nodes(edge_index, node_count, seed_nodes, hops=2, max_nodes=512):
    neighbors = build_adjacency(edge_index, node_count)
    seed_nodes = [int(n) for n in seed_nodes if 0 <= int(n) < node_count]
    if not seed_nodes:
        return np.arange(min(node_count, max_nodes), dtype=np.int64)

    reached = set(seed_nodes)
    frontier = set(seed_nodes)
    ordered = list(dict.fromkeys(seed_nodes))

    for _ in range(hops):
        next_frontier = set()
        for node in sorted(frontier):
            for nbr in sorted(neighbors[node]):
                if nbr not in reached:
                    reached.add(nbr)
                    next_frontier.add(nbr)
                    ordered.append(nbr)
        frontier = next_frontier
        if not frontier:
            break

    # Always keep configurable nodes first, then nearest discovered neighbours.
    if len(ordered) > max_nodes:
        seed_set = list(dict.fromkeys(seed_nodes))
        remaining = [node for node in ordered if node not in set(seed_set)]
        ordered = seed_set + remaining[:max(0, max_nodes - len(seed_set))]
    return np.array(sorted(ordered), dtype=np.int64)


def induced_edges(edge_index, kept_nodes):
    kept_nodes = np.asarray(kept_nodes, dtype=np.int64)
    old_to_new = {int(old): i for i, old in enumerate(kept_nodes)}
    src_list = []
    dst_list = []
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if int(src) in old_to_new and int(dst) in old_to_new:
            src_list.append(old_to_new[int(src)])
            dst_list.append(old_to_new[int(dst)])
            src_list.append(old_to_new[int(dst)])
            dst_list.append(old_to_new[int(src)])
    for i in range(len(kept_nodes)):
        src_list.append(i)
        dst_list.append(i)
    return np.asarray(src_list, dtype=np.int64), np.asarray(dst_list, dtype=np.int64), old_to_new


def inspect_subgraph_sizes(max_files=5):
    rows = []
    for collection_name in GNN_COLLECTIONS:
        for file_path in split_files(collection_name, "train")[:max_files]:
            with np.load(file_path) as data:
                node_count = int(data["node_feat"].shape[0])
                node_config_ids = np.asarray(data["node_config_ids"], dtype=np.int64)
                kept = khop_subgraph_nodes(data["edge_index"], node_count, node_config_ids, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)
                rows.append({
                    "collection": collection_name,
                    "file": file_path.stem,
                    "node_count": node_count,
                    "configurable_nodes": len(node_config_ids),
                    "subgraph_nodes": len(kept),
                })
    return pd.DataFrame(rows)


display(inspect_subgraph_sizes())


## 6. GraphSAGE Ranker

The model outputs one scalar score per configuration. Lower score means faster predicted runtime.


In [ ]:
class GraphSAGERanker(nn.Module):
    def __init__(self, node_dim, config_dim, hidden_dim=GNN_HIDDEN_DIM, opcode_vocab_size=256, opcode_emb_dim=16):
        super().__init__()
        self.opcode_embedding = nn.Embedding(opcode_vocab_size, opcode_emb_dim)
        self.input_proj = nn.Linear(node_dim + config_dim + opcode_emb_dim, hidden_dim)
        self.sage1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.sage2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden_dim, 1),
        )

    def sage_step(self, h, edge_src, edge_dst, degree, layer):
        neigh = torch.zeros_like(h)
        neigh.index_add_(1, edge_dst, h[:, edge_src, :])
        neigh = neigh / degree.view(1, -1, 1).clamp_min(1.0)
        return F.relu(layer(torch.cat([h, neigh], dim=-1))) + h

    def forward(self, base_node, opcode, config_node, edge_src, edge_dst, degree, config_mask):
        batch_size = config_node.shape[0]
        base = base_node.unsqueeze(0).expand(batch_size, -1, -1)
        opcode_emb = self.opcode_embedding(opcode).unsqueeze(0).expand(batch_size, -1, -1)
        h = F.relu(self.input_proj(torch.cat([base, config_node, opcode_emb], dim=-1)))
        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage1)
        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage2)

        global_pool = torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1)
        mask = config_mask.view(1, -1, 1).float()
        denom = mask.sum(dim=1).clamp_min(1.0)
        config_mean = (h * mask).sum(dim=1) / denom
        config_max = h.masked_fill(mask == 0, -1e9).max(dim=1).values
        config_max = torch.where((mask.sum(dim=1) > 0), config_max, torch.zeros_like(config_max))
        config_pool = torch.cat([config_mean, config_max], dim=-1)
        return self.head(torch.cat([global_pool, config_pool], dim=-1)).squeeze(-1)


def pairwise_margin_ranking_loss(scores, runtimes, margin=PAIR_MARGIN, min_gap=PAIR_MIN_LOG_RUNTIME_GAP, max_pairs=PAIR_MAX_PAIRS):
    log_runtime = torch.log1p(runtimes)
    diff = log_runtime.view(-1, 1) - log_runtime.view(1, -1)
    pairs = torch.nonzero(diff < -min_gap, as_tuple=False)
    if pairs.numel() == 0:
        return F.smooth_l1_loss(scores, log_runtime - log_runtime.median())
    if len(pairs) > max_pairs:
        idx = torch.randperm(len(pairs), device=scores.device)[:max_pairs]
        pairs = pairs[idx]
    fast = pairs[:, 0]
    slow = pairs[:, 1]
    return F.relu(margin + scores[fast] - scores[slow]).mean()


## 7. Model Wrapper

In [ ]:
class PreparedGraphCache:
    """One-time streaming extraction plus an LRU of static CPU graph tensors.

    Disk entries are keyed by resolved source path, size, and modification time.
    Change/remove GNN_CACHE_DIR if a source was replaced while preserving both.
    No labels or model outputs participate in the cache key or graph features.
    """
    def __init__(self, root=GNN_CACHE_DIR, max_mb=GNN_GRAPH_CACHE_MB):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = int(max_mb * 1024**2)
        self.entries = OrderedDict()
        self.bytes = 0
        self.extractions = 0
        self.graph_preparations = 0

    def get(self, file_path, ranker):
        path = Path(file_path).resolve()
        stat = path.stat()
        token = f"{path}|{stat.st_size}|{stat.st_mtime_ns}"
        key = hashlib.sha256(token.encode()).hexdigest()[:24]
        graph_key = (key, GNN_SUBGRAPH_HOPS, GNN_MAX_SUBGRAPH_NODES)
        if graph_key in self.entries:
            self.entries.move_to_end(graph_key)
            return self.entries[graph_key]

        config_path = self.root / f"{key}-node_config_feat.npy"
        if not config_path.exists():
            temporary = config_path.with_suffix(".tmp")
            try:
                # Stream the .npy member to disk: do not expand the whole array in RAM.
                with zipfile.ZipFile(path) as archive:
                    with archive.open("node_config_feat.npy") as src, temporary.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                temporary.replace(config_path)
                self.extractions += 1
            finally:
                temporary.unlink(missing_ok=True)
        configs = np.load(config_path, mmap_mode="r", allow_pickle=False)
        if configs.ndim != 3 or configs.dtype.hasobject:
            raise ValueError(f"Invalid node_config_feat shape/dtype: {path}")
        with np.load(path, allow_pickle=False) as archive:
            # Each small/static member is decompressed only once per cache miss.
            data = {name: archive[name] for name in
                    ["node_feat", "node_opcode", "edge_index", "node_config_ids"]}
            runtime = archive["config_runtime"] if "config_runtime" in archive else None
        data["node_config_feat"] = configs
        ranker.ensure_model(data)
        graph = ranker.prepare_graph(data)
        # Cache CPU tensors, never retain all graphs in accelerator memory.
        graph = tuple(x.cpu() if torch.is_tensor(x) else x for x in graph)
        nbytes = sum(x.numel() * x.element_size() if torch.is_tensor(x) else x.nbytes for x in graph)
        if runtime is not None:
            if len(runtime) != len(configs):
                raise ValueError(f"Runtime/configuration count mismatch: {path}")
            nbytes += runtime.nbytes
        entry = {"graph": graph, "configs": configs, "runtime": runtime, "bytes": nbytes}
        self.graph_preparations += 1
        while self.entries and self.bytes + nbytes > self.max_bytes:
            _, old = self.entries.popitem(last=False)
            self.bytes -= old["bytes"]
        if nbytes <= self.max_bytes:
            self.entries[graph_key] = entry
            self.bytes += nbytes
        return entry


class LayoutGNNRanker:
    def __init__(self):
        self.device = DEVICE
        self.model = None
        self.node_dim = None
        self.config_dim = None
        self.history = []
        self.cache = PreparedGraphCache()
        self.best_epoch = None

    def base_node_features(self, data, kept_nodes):
        node_feat = np.asarray(data["node_feat"][kept_nodes], dtype=np.float32)
        node_feat = np.log1p(np.maximum(node_feat, 0.0))
        return (node_feat - node_feat.mean(axis=0, keepdims=True)) / (node_feat.std(axis=0, keepdims=True) + 1e-6)

    def prepare_graph(self, data):
        node_count = int(data["node_feat"].shape[0])
        raw_config_nodes = np.asarray(data["node_config_ids"], dtype=np.int64)
        kept_nodes = khop_subgraph_nodes(data["edge_index"], node_count, raw_config_nodes, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)
        edge_src_np, edge_dst_np, old_to_new = induced_edges(data["edge_index"], kept_nodes)

        config_local_pairs = [(pos, old_to_new[int(old)]) for pos, old in enumerate(raw_config_nodes) if int(old) in old_to_new]
        config_positions = np.array([p for p, _ in config_local_pairs], dtype=np.int64)
        config_local_nodes = np.array([n for _, n in config_local_pairs], dtype=np.int64)
        config_mask = np.zeros(len(kept_nodes), dtype=np.float32)
        config_mask[config_local_nodes] = 1.0

        degree = np.bincount(edge_dst_np, minlength=len(kept_nodes)).astype(np.float32)
        base_node = torch.tensor(self.base_node_features(data, kept_nodes), dtype=torch.float32, device=self.device)
        opcode = np.clip(np.asarray(data["node_opcode"][kept_nodes], dtype=np.int64), 0, 255)
        opcode = torch.tensor(opcode, dtype=torch.long, device=self.device)
        edge_src = torch.tensor(edge_src_np, dtype=torch.long, device=self.device)
        edge_dst = torch.tensor(edge_dst_np, dtype=torch.long, device=self.device)
        degree = torch.tensor(degree, dtype=torch.float32, device=self.device)
        config_mask = torch.tensor(config_mask, dtype=torch.float32, device=self.device)
        return base_node, opcode, edge_src, edge_dst, degree, config_mask, config_positions, config_local_nodes

    def ensure_model(self, data):
        node_dim = int(data["node_feat"].shape[1])
        config_dim = int(data["node_config_feat"].shape[2])
        if self.model is None:
            self.node_dim = node_dim
            self.config_dim = config_dim
            self.model = GraphSAGERanker(node_dim=node_dim, config_dim=config_dim).to(self.device)
        if node_dim != self.node_dim or config_dim != self.config_dim:
            raise ValueError("Inconsistent node/config feature dimensions")
        return self.model

    def config_tensor(self, data, config_indices, n_nodes, config_positions, config_local_nodes):
        selected = np.asarray(data["node_config_feat"][config_indices], dtype=np.float32)
        selected = np.where(selected == -1, 0.0, selected)
        config_node = np.zeros((len(config_indices), n_nodes, selected.shape[2]), dtype=np.float32)
        if len(config_positions):
            config_node[:, config_local_nodes, :] = selected[:, config_positions, :]
        return torch.tensor(config_node, dtype=torch.float32, device=self.device)

    def device_graph(self, entry):
        return tuple(x.to(self.device) if torch.is_tensor(x) else x for x in entry["graph"])

    def cached_config_tensor(self, entry, config_indices, graph):
        base_node, _, _, _, _, _, positions, local_nodes = graph
        # Select configurations AND retained nodes before converting to float32.
        # Only this batch is copied out of the memory map.
        selected = np.asarray(entry["configs"][np.ix_(config_indices, positions)], dtype=np.float32)
        selected = np.where(selected == -1, 0.0, selected)
        padded = np.zeros((len(config_indices), len(base_node), entry["configs"].shape[2]), dtype=np.float32)
        padded[:, local_nodes, :] = selected
        return torch.from_numpy(padded).to(self.device)

    def forward_batch(self, entry, indices, graph):
        base, opcode, src, dst, degree, mask, _, _ = graph
        config = self.cached_config_tensor(entry, indices, graph)
        return self.model(base, opcode, config, src, dst, degree, mask)

    def fit(self, collection_name, files, valid_files=None):
        files = list(files)
        valid_files = list(valid_files or [])
        if not files:
            raise ValueError(f"No training files for {collection_name}")
        if {Path(p).resolve() for p in files} & {Path(p).resolve() for p in valid_files}:
            raise ValueError("Training and validation files overlap")
        torch.manual_seed(RANDOM_SEED)
        started = time.perf_counter()
        # Warm the cache explicitly so the startup cost is visible.
        for path in tqdm(files + valid_files, desc="Prepare graph cache", unit="file"):
            self.cache.get(path, self)
        preparation_seconds = time.perf_counter() - started
        print(f"Cache preparation: {preparation_seconds:.2f}s; "
              f"extractions={self.cache.extractions}; static RAM={self.cache.bytes / 1024**2:.1f} MiB")
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=GNN_LR, weight_decay=1e-4)
        best_score, best_state, stale = -np.inf, None, 0
        run_dir = GNN_OUTPUT_DIR / collection_name.replace(":", "_")
        run_dir.mkdir(parents=True, exist_ok=True)

        for epoch in range(1, GNN_EPOCHS + 1):
            self.model.train()  # Validation switches to eval mode after each epoch.
            losses, examples = [], 0
            prepare_seconds = 0.0
            tensor_seconds = 0.0
            step_seconds = 0.0
            epoch_start = time.perf_counter()
            progress = tqdm(files, desc=f"{collection_name} epoch {epoch}/{GNN_EPOCHS}", unit="file")
            for file_id, file_path in enumerate(progress):
                t = time.perf_counter()
                entry = self.cache.get(file_path, self)
                graph = self.device_graph(entry)
                if entry["runtime"] is None:
                    raise ValueError(f"Training runtime labels missing: {file_path}")
                # Keep the original fixed sample for the speed comparison.
                indices = choose_runtime_stratified_indices(entry["runtime"], GNN_MAX_TRAIN_CONFIGS_PER_FILE,
                                                           seed=RANDOM_SEED + file_id)
                np.random.default_rng(RANDOM_SEED + epoch + file_id).shuffle(indices)
                prepare_seconds += time.perf_counter() - t
                for start in range(0, len(indices), GNN_BATCH_SIZE):
                    batch = indices[start:start + GNN_BATCH_SIZE]
                    t = time.perf_counter()
                    config = self.cached_config_tensor(entry, batch, graph)
                    runtime = torch.as_tensor(np.asarray(entry["runtime"][batch], dtype=np.float32), device=self.device)
                    tensor_seconds += time.perf_counter() - t
                    t = time.perf_counter()
                    base, opcode, src, dst, degree, mask, _, _ = graph
                    scores = self.model(base, opcode, config, src, dst, degree, mask)
                    loss = pairwise_margin_ranking_loss(scores, runtime)
                    if not torch.isfinite(loss):
                        raise ValueError(f"Nonfinite training loss: {file_path}")
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    losses.append(float(loss.detach().cpu()))
                    step_seconds += time.perf_counter() - t
                    examples += len(batch)
                progress.set_postfix(loss=np.mean(losses[-10:]))
            train_seconds = time.perf_counter() - epoch_start
            row = {"epoch": epoch, "loss": float(np.mean(losses)),
                   "cache_preparation_seconds": preparation_seconds if epoch == 1 else 0.0,
                   "graph_lookup_seconds": prepare_seconds, "batch_tensor_seconds": tensor_seconds,
                   "model_step_seconds": step_seconds, "train_seconds": train_seconds,
                   "configs_per_second": examples / max(train_seconds, 1e-9)}
            if valid_files and (epoch % GNN_VALIDATE_EVERY == 0 or epoch == GNN_EPOCHS):
                t = time.perf_counter()
                valid = validate_model(self, collection_name, files=valid_files)
                score = float(valid["ranking_score"].mean())
                row.update(validation_score=score, validation_seconds=time.perf_counter() - t)
                if np.isfinite(score) and score > best_score + GNN_MIN_DELTA:
                    best_score, stale = score, 0
                    self.best_epoch = epoch
                    best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                    torch.save({"model_state": best_state, "epoch": epoch, "validation_score": score,
                                "node_dim": self.node_dim, "config_dim": self.config_dim,
                                "hidden_dim": GNN_HIDDEN_DIM, "seed": RANDOM_SEED,
                                "subgraph_hops": GNN_SUBGRAPH_HOPS, "max_nodes": GNN_MAX_SUBGRAPH_NODES,
                                "batch_size": GNN_BATCH_SIZE,
                                "train_files": [str(p) for p in files],
                                "valid_files": [str(p) for p in valid_files]}, run_dir / "best.pt")
                else:
                    stale += 1
            self.history.append(row)
            pd.DataFrame(self.history).to_csv(run_dir / "history.csv", index=False)
            print(row)
            if valid_files and stale >= GNN_PATIENCE:
                print(f"Early stopping; best epoch: {self.best_epoch}")
                break
        if best_state is not None:
            self.model.load_state_dict(best_state)
        self.model.eval()
        return self

    def predict_file(self, file_path, split="valid", max_configs=None, seed=RANDOM_SEED):
        if self.model is None:
            raise RuntimeError("Fit or load a model before prediction")
        self.model.eval()
        entry = self.cache.get(file_path, self)
        indices = choose_indices(len(entry["configs"]), max_items=max_configs, seed=seed)
        graph = self.device_graph(entry)
        preds = []
        with torch.inference_mode():
            for start in range(0, len(indices), GNN_PREDICT_BATCH_SIZE):
                batch = indices[start:start + GNN_PREDICT_BATCH_SIZE]
                preds.append(self.forward_batch(entry, batch, graph).cpu().numpy())
        return indices, np.concatenate(preds) if preds else np.array([], dtype=np.float64)


## 8. Train and Validate Layout GNNs

Each epoch reports CPU time for graph lookup, batch construction, model updates and
validation. The best checkpoint is restored after training. Epoch timing categories
are intended for CPU; asynchronous GPU execution requires separate profiling.

Validation samples configurations uniformly without inspecting runtime labels. Its
sampled concordance diagnostic gives prediction ties zero credit. It is not the
official Kaggle scorer, and is **not directly comparable to the old runtime-stratified
validation scores**. Evaluate the tree baseline on the same graph/configuration IDs
before drawing conclusions. MAE is omitted because pairwise scores have no calibrated
runtime scale. Final selection should use the official metric on full configurations.


In [ ]:
def validate_model(model, collection_name, files=None):
    if files is None:
        files = split_files(collection_name, "valid")[:GNN_MAX_VALID_FILES]
    rows = []
    for i, file_path in enumerate(files):
        indices, pred = model.predict_file(file_path, split="valid",
                                          max_configs=GNN_MAX_VALID_CONFIGS_PER_FILE, seed=RANDOM_SEED + i)
        entry = model.cache.get(file_path, model)
        if entry["runtime"] is None:
            raise ValueError(f"Validation runtime labels missing: {file_path}")
        truth = np.asarray(entry["runtime"][indices], dtype=np.float64)
        if not (np.isfinite(truth).all() and (truth > 0).all() and np.isfinite(pred).all()):
            raise ValueError(f"Invalid validation runtimes or predictions: {file_path}")
        rows.append({"collection": collection_name, "file": file_path.stem,
                     "n_configs": len(truth),
                     "ranking_score": sampled_kendall_score(truth, pred, seed=RANDOM_SEED + i)})
    return pd.DataFrame(rows)


models = {}
validation_tables = []

for collection_name in GNN_COLLECTIONS:
    train_files = split_files(collection_name, "train")[:GNN_MAX_TRAIN_FILES]
    print("\n" + "=" * 80)
    print("Training GNN:", collection_name)
    print("train files:", len(train_files))
    start = time.perf_counter()
    valid_files = split_files(collection_name, "valid")[:GNN_MAX_VALID_FILES]
    model = LayoutGNNRanker().fit(collection_name, train_files, valid_files=valid_files)
    fit_seconds = time.perf_counter() - start
    models[collection_name] = model
    valid_df = validate_model(model, collection_name)
    valid_df["train_seconds"] = round(fit_seconds, 2)
    validation_tables.append(valid_df)
    display(valid_df)

validation_df = pd.concat(validation_tables, ignore_index=True)
summary = validation_df.groupby("collection", as_index=False).agg(
    valid_files=("file", "count"),
    ranking_score=("ranking_score", "mean"),
    fit_seconds_including_validation=("train_seconds", "max"),
)
display(summary)


## 9. CPU Experiment Order

1. Run three epochs with the same 8 graphs / 64 configurations per graph.
2. Compare batch sizes 4, 8 and 16 and CPU thread counts 1, 2 and 4 on your machine.
   Record startup time separately from warm training and validation time. Changing
   batch size changes ranking pairs and optimizer steps, so recheck score too.
3. If validation improves, raise `GNN_EPOCHS` to 10 or 20. Patience defaults to three
   validation checks. The best model is restored even if the last epoch is worse.
4. Increase graph/configuration coverage next. The initial training subset remains
   fixed for a controlled speed comparison; more epochs do not add new data.
5. Compare against the baseline using identical validation IDs, then combine only
   independently validated gains with the other three collections.

Graph tensors use a bounded CPU LRU (256 MiB by default). Evicted graphs are rebuilt,
but extracted configuration arrays remain on disk. Configurable nodes are retained
even when their count exceeds the 512-node neighbourhood target; reduce batch size
if such a graph exceeds available RAM. No competitor code or predictions are used.
